In [ ]:
# Imports for agent and tools usage, and Excel data loading
import langchain_core
import pandas as pd
import matplotlib.pyplot as plt
import difflib
from typing import List, Dict, Any
from langchain.agents import initialize_agent, AgentType
from langchain.prompts import load_prompt
from langchain_community.chat_models import ChatOpenAI, ChatOllama
import importlib
from agents import ReActAgent
# Use StrOutputParser to extract plain string output from ChatOllama
from langchain_core.output_parsers.string import StrOutputParser
from prompts import react_prompt_meta
# To run this code you need to install the following dependencies:
# pip install google-genai

import base64
import os
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
  model="gpt-4.1",
  input=[
    {
      "role": "system",
      "content": [
        {
          "type": "input_text",
          "text": "You are an expert"
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "input_text",
          "text": "what is the square root of some number"
        }
      ]
    },
    {
      "role": "assistant",
      "content": [
        {
          "type": "output_text",
          "text": "The **square root** of a number \\( x \\), written as \\( \\sqrt{x} \\), is the value \\( y \\) such that \\( y \\times y = x \\).\n\n### For example:\n- The square root of **9** is **3**, because \\( 3 \\times 3 = 9 \\).\n- The square root of **16** is **4**, because \\( 4 \\times 4 = 16 \\).\n- The square root of **25** is **5**, because \\( 5 \\times 5 = 25 \\).\n\n### Properties:\n- Every non-negative number (0 or greater) has a **non-negative square root** (called the **principal square root**).\n- Negative numbers do **"
        }
      ]
    }
  ],
  text={
    "format": {
      "type": "text"
    }
  },
  reasoning={},
  tools=[],
  temperature=1,
  max_output_tokens=2048,
  top_p=1,
  store=True
)


ModuleNotFoundError: No module named 'google.generativeai'

Type:        module
String form: <module 'google' (namespace) from ['/home/nitish/Documents/github/PublicReportResearch/.venv/lib/python3.12/site-packages/google']>
Docstring:   <no docstring>

In [9]:
llm = ChatOllama(model="qwen2.5:7b", temperature=0.5)

In [15]:
# Define a simple Pydantic schema and use ChatOllama for structured output

from pydantic import BaseModel, Field
from langchain.output_parsers import PydanticOutputParser

# Define a simple schema
class CompanyInfo(BaseModel):
    name: str = Field(..., description="The name of the company")
    revenue: float = Field(..., description="The revenue of the company in millions USD")
    employees: int = Field(..., description="Number of employees")

# Create a parser for the schema
parser = PydanticOutputParser(pydantic_object=CompanyInfo)

# Example prompt
prompt = (
    "Extract the following information about Apple Inc.: "
    "Company name, revenue in millions USD, and number of employees. "
    f"Format your answer as: {parser.get_format_instructions()}"
)

str_parser = StrOutputParser()


# Use ChatOllama to get structured output
response = llm.invoke(prompt)
parsed_str = str_parser.parse(response)

parsed = parser.parse(response)
print(parsed)

ValidationError: 1 validation error for Generation
text
  Input should be a valid string [type=string_type, input_value=AIMessage(content='```jso...7b-9123-a9f79937635d-0'), input_type=AIMessage]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type

In [14]:
response | str_parser

TypeError: Expected a Runnable, callable or dict.Instead got an unsupported type: <class 'langchain_core.messages.ai.AIMessage'>

In [5]:
df = pd.read_excel("12_metrics.xlsx")
available_metrics = df.columns.tolist()[2:]
df.head()

,Datetime,CompanyName,TotalRevenue,InterestIncome,NonInterestIncome,InterestExpense,NetInterestIncome,ProvisionForLoanLosses,NonInterestExpense,NetIncome,EarningsPerShare,TotalAssets,TotalLoans,TotalDeposits,ShareholdersEquity
0,2019-03-31,AMERICAN EXPRESS COMPANY,6.697000e+09,2.725000e+09,8.305000e+09,8.950000e+08,2.059000e+09,5.250000e+08,7.597000e+09,1.550000e+09,1.81,1.971930e+11,NaN,7.285700e+10,22218000000
1,2019-06-30,AMERICAN EXPRESS COMPANY,1.377600e+10,5.489000e+09,1.706900e+10,1.786000e+09,4.133000e+09,1.128000e+09,1.535500e+10,3.311000e+09,3.88,1.976030e+11,NaN,7.259000e+10,23092000000
2,2019-09-30,AMERICAN EXPRESS COMPANY,2.082700e+10,8.374000e+09,2.585500e+10,2.663000e+09,6.336000e+09,1.732000e+09,2.319900e+10,5.066000e+09,5.97,1.941840e+11,NaN,7.329800e+10,23025000000
3,2019-12-31,AMERICAN EXPRESS COMPANY,2.815900e+10,1.130800e+10,3.493600e+10,3.464000e+09,8.620000e+09,2.462000e+09,3.155400e+10,6.759000e+09,8.00,1.983210e+11,NaN,7.328700e+10,23071000000
4,2020-03-31,AMERICAN EXPRESS COMPANY,6.296000e+09,2.909000e+09,7.980000e+09,7.160000e+08,2.330000e+09,1.876000e+09,7.237000e+09,3.670000e+08,0.41,1.860600e+11,NaN,7.796200e+10,21006000000


In [9]:
from tools import compare_companies, compare_within_one_company

tools = [
    {
        "name": "compare_companies",
        "func": compare_companies,
        "description": "Compare specific metrics for selected companies in a given quarter."
    },
    {
        "name": "compare_within_one_company",
        "func": compare_within_one_company,
        "description": "Compare specific metrics for a single company across multiple quarters."
    }
]

agent = ReActAgent(prompt_from_hub = True,
    tools=[t["func"] for t in tools],
    llm=llm
)



TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'bool'

### ReAct Variations

In [1]:
# First version with one set of tools and the same template prompt
agent_prompt = react_prompt_meta

print(agent_prompt)

NameError: name 'react_prompt_meta' is not defined